# 97 EP channel geometry window

Channel $\qtyrange{3}{50}{\milli\metre}$ × $\qtyrange{1}{40}{\milli\metre}$, spacing $\qtyrange{10}{50}{\milli\metre}$, Fe $\qtyrange{1}{7}{\gram\per\centi\metre\squared}$, VF $0.5$–$0.9$.


$$
\newcommand{\per}{/}
\newcommand{\squared}{^{2}}
\newcommand{\cubed}{^{3}}
\newcommand{\of}[1]{\,\mathrm{#1}}
\newcommand{\milli}{\mathrm{m}}
\newcommand{\centi}{\mathrm{c}}
\newcommand{\micro}{\mathrm{\mu}}
\newcommand{\kilo}{\mathrm{k}}
\newcommand{\metre}{\mathrm{m}}
\newcommand{\gram}{\mathrm{g}}
\newcommand{\kilogram}{\mathrm{kg}}
\newcommand{\second}{\mathrm{s}}
\newcommand{\minute}{\mathrm{min}}
\newcommand{\hour}{\mathrm{h}}
\newcommand{\day}{\mathrm{d}}
\newcommand{\ampere}{\mathrm{A}}
\newcommand{\volt}{\mathrm{V}}
\newcommand{\watt}{\mathrm{W}}
\newcommand{\ohm}{\mathrm{\Omega}}
\newcommand{\kelvin}{\mathrm{K}}
\newcommand{\celsius}{{}^{\circ}\mathrm{C}}
\newcommand{\coulomb}{\mathrm{C}}
\newcommand{\joule}{\mathrm{J}}
\newcommand{\pascal}{\mathrm{Pa}}
\newcommand{\litre}{\mathrm{L}}
\newcommand{\mole}{\mathrm{mol}}
\newcommand{\molar}{\mathrm{M}}
\newcommand{\percent}{\%}
\newcommand{\atm}{\mathrm{atm}}
\newcommand{\newton}{\mathrm{N}}
\newcommand{\qty}[2]{#1\,#2}
\newcommand{\qtyrange}[3]{#1\text{--}#2\,#3}
\newcommand{\unit}[1]{#1}
\newcommand{\si}[1]{#1}
$$


## Governing equations

Quantities in this scenario use siunitx ($\qty$, $\unit$, $\qtyrange$).
Default electrolyte $\qty{6}{\molar}$ KOH at $\qty{303}{\kelvin}$ unless noted.
Patents: US12308414B2, EP4602674A1.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params
from plant_sim.siunitx_nb import SIUNITX_PREAMBLE

In [ ]:
from IPython import get_ipython
from IPython.display import Markdown, display

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
display(Markdown(SIUNITX_PREAMBLE))
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation

In [ ]:
from plant_sim.substructures.channeled_electrode import ChanneledElectrode, assert_channel_window

assert_channel_window(P.channel_length_m, P.channel_width_m, P.channel_spacing_m, P)
ch = ChanneledElectrode(P)
cmp = ch.compare_layouts(P.I_discharge_100h_A * 8)
print(cmp)
ce_frame = pd.DataFrame(
    {
        "layout": [k[3:] for k in cmp if k.startswith("CE_")],
        "CE": [v for k, v in cmp.items() if k.startswith("CE_")],
    }
)
fig, ax = plt.subplots()
sns.barplot(data=ce_frame, x="layout", y="CE", ax=ax)
ax.set_xlabel("layout")
ax.set_ylabel("CE")
ax.set_title("EP channel window: charge CE by layout")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
print(
    "loading",
    P.loading_g_cm2,
    "VF",
    P.vf_electrolyte_charged,
    "chan",
    P.channel_length_m,
    P.channel_width_m,
    P.channel_spacing_m,
)

## Verification against patents / SSS002

In [ ]:
print("finite trajectories and patent-window parameters: see printed checks above")